# 🏆 Deep Past Challenge - Baseline + Extended Dataset
## Translating Ancient Akkadian Cuneiform to English

---

### ⭐ If this notebook helps you, please upvote! ⭐

---

## 📜 Competition Background

The **Deep Past Challenge** aims to translate 4,000-year-old Akkadian cuneiform tablets into English. These tablets contain commercial records from ancient Assyrian merchants.

| Fact | Detail |
|------|--------|
| 🌍 Language | Old Assyrian (Akkadian dialect) |
| 📅 Period | ca. 1950-1850 BC |
| 📝 Content | Commercial letters, contracts |
| 📊 Training Data | 1,561 parallel examples |
| 💰 Prize Pool | $50,000 |

## 🔍 Our Approach: Retrieval-Based Translation

With only **1,561 training examples**, neural MT struggles significantly:
- [UC Berkeley's CuneiTranslate](https://www.ischool.berkeley.edu/projects/2024/cuneitranslate-unlocking-ancient-mesopotamian-knowledge) tried T5, mT5, NLLB → **BLEU < 8%**
- Small dataset = severe overfitting for neural models

We use **retrieval-based translation** instead:
1. 🔎 **Find Similar Texts**: TF-IDF with character n-grams
2. ✂️ **Segment Alignment**: Proportional extraction based on line numbers
3. 📐 **Boundary Refinement**: Adjust to sentence boundaries

## 💡 Key Discovery

All 4 test samples come from the **same ancient text**, with an **87%+ similarity match** in training data!

This high similarity makes retrieval-based methods highly effective for this specific competition.

## 📊 Extended Dataset

This notebook also shows how to use our **Extended Dataset** (7,953 texts) for improved TF-IDF matching!

---

## 1. 📦 Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import os
import warnings
warnings.filterwarnings('ignore')

# Kaggle/Local path detection
if os.path.exists('/kaggle/input/deep-past-initiative-machine-translation'):
    DATA_PATH = '/kaggle/input/deep-past-initiative-machine-translation/'
elif os.path.exists('/kaggle/input'):
    # Find the competition data folder
    for folder in os.listdir('/kaggle/input'):
        if 'deep-past' in folder.lower() or 'machine-translation' in folder.lower():
            DATA_PATH = f'/kaggle/input/{folder}/'
            break
    else:
        DATA_PATH = '/kaggle/input/'
else:
    DATA_PATH = './data/'

print(f"Data path: {DATA_PATH}")

# Load datasets with error handling
try:
    train_df = pd.read_csv(f'{DATA_PATH}train.csv')
    test_df = pd.read_csv(f'{DATA_PATH}test.csv')
    print(f"Training samples: {len(train_df):,}")
    print(f"Test samples: {len(test_df)}")
except Exception as e:
    print(f"Error loading data: {e}")
    # List available files
    if os.path.exists(DATA_PATH):
        print(f"Files in {DATA_PATH}: {os.listdir(DATA_PATH)}")
    raise

## 2. 🔎 Analyze Test Data Structure

Understanding the test data structure is **crucial** for this competition.

In [ ]:
print("Test Data Structure:")
print("-" * 50)
print(f"Unique text IDs: {test_df['text_id'].nunique()}")
print(f"Text ID: {test_df['text_id'].iloc[0]}")
print()

for _, row in test_df.iterrows():
    print(f"ID {row['id']}: Lines {row['line_start']:2d}-{row['line_end']:2d} | {len(row['transliteration']):3d} chars")

print()
print("KEY INSIGHT: All test samples are segments of the SAME text!")

## 3. 🧮 Build TF-IDF Model

We use **character n-grams** (2-6 characters) because:
- Akkadian has rich morphology with many word variations
- Character-level features capture spelling similarities
- Sublinear TF scaling (log) prevents common patterns from dominating
- BPE tokenization (used in neural MT) struggles with limited vocab

> 📚 **Research Context**: 
> - [PNAS Nexus 2023](https://academic.oup.com/pnasnexus/article/2/5/pgad096/7147349) achieved BLEU4 37.47 using CNN + BPE (on larger ORACC corpus)
> - [Berkeley CuneiTranslate](https://www.ischool.berkeley.edu/projects/2024/cuneitranslate-unlocking-ancient-mesopotamian-knowledge) found PyTorch BPE effective for low-resource settings

In [ ]:
# Combine all test segments into one text for overall matching
full_test_text = ' '.join(test_df['transliteration'].tolist())

# Build TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    analyzer='char_wb',      # Character n-grams with word boundaries
    ngram_range=(2, 6),      # 2 to 6 character sequences
    max_features=25000,      # Vocabulary size
    sublinear_tf=True        # Use log(TF) for better scaling
)

# Fit on training data and transform
train_vectors = vectorizer.fit_transform(train_df['transliteration'].str.lower())
test_vector = vectorizer.transform([full_test_text.lower()])

print(f"Vocabulary size: {len(vectorizer.vocabulary_):,}")
print(f"Train matrix: {train_vectors.shape}")

## 4. 🎯 Find Best Matching Training Sample

We compute cosine similarity between the combined test text and all training samples.

In [ ]:
# Compute similarities
similarities = cosine_similarity(test_vector, train_vectors)[0]

# Get top matches
top_k = 5
top_indices = similarities.argsort()[-top_k:][::-1]

print(f"Top {top_k} Most Similar Training Samples:")
print("=" * 60)

for rank, idx in enumerate(top_indices, 1):
    sim = similarities[idx]
    trans_preview = train_df.iloc[idx]['transliteration'][:80]
    print(f"\n{rank}. Similarity: {sim:.4f} ({sim*100:.1f}%)")
    print(f"   {trans_preview}...")

In [ ]:
# Extract best match
best_idx = top_indices[0]
best_similarity = similarities[best_idx]
best_transliteration = train_df.iloc[best_idx]['transliteration']
best_translation = train_df.iloc[best_idx]['translation']

print(f"Best Match Similarity: {best_similarity:.4f} ({best_similarity*100:.1f}%)")
print()
print("=" * 60)
print("BEST MATCH - TRANSLITERATION (Akkadian)")
print("=" * 60)
print(best_transliteration)
print()
print("=" * 60)
print("BEST MATCH - TRANSLATION (English)")
print("=" * 60)
print(best_translation)

## 5. ✂️ Segment Translation by Line Proportions

Since test samples are line-based segments of the same text, we:
1. Calculate proportional position of each segment (based on line numbers)
2. Extract corresponding portion from the best-matching translation
3. Adjust boundaries to align with sentence endings

In [ ]:
def extract_translation_segment(translation, line_start, line_end, total_lines):
    """
    Extract a proportional segment from the translation based on line numbers.
    Adjusts boundaries to align with sentence endings for cleaner output.
    """
    if not translation or total_lines <= 0:
        return translation if translation else "Translation not available."
    
    # Calculate proportional character positions
    start_ratio = max(0, (line_start - 1) / total_lines)
    end_ratio = min(1, line_end / total_lines)

    orig_start = int(len(translation) * start_ratio)
    orig_end = int(len(translation) * end_ratio)

    start_char = orig_start
    end_char = orig_end

    # Adjust start to nearest sentence boundary (look back up to 150 chars)
    if start_char > 0:
        search_start = max(0, start_char - 150)
        last_period = translation.rfind('.', search_start, start_char)
        if last_period > 0:
            start_char = last_period + 2

    # Adjust end to nearest sentence boundary (look forward up to 150 chars)
    if end_char < len(translation):
        search_end = min(len(translation), end_char + 150)
        next_period = translation.find('.', end_char, search_end)
        if next_period > 0:
            end_char = next_period + 1
        else:
            # If no period found, try to end at a space (word boundary)
            space_pos = translation.find(' ', end_char, search_end)
            if space_pos > 0:
                end_char = space_pos

    # Safety check: ensure start < end
    if start_char >= end_char:
        start_char = orig_start
        end_char = orig_end

    # Ensure we don't cut off mid-word at start
    if start_char > 0 and start_char < len(translation):
        while start_char > 0 and translation[start_char-1] not in ' .\n':
            start_char -= 1

    # Ensure we don't cut off mid-word at end  
    if end_char < len(translation):
        while end_char < len(translation) and translation[end_char] not in ' .\n':
            end_char += 1

    segment = translation[start_char:end_char].strip()

    # Fallbacks
    if not segment:
        segment = translation[orig_start:orig_end].strip()
    if not segment:
        segment = translation.strip()

    return segment

print("Segmentation function ready!")

## 6. 🚀 Generate Translations

In [ ]:
# Generate translations with robust error handling
print("=" * 70)
print("GENERATING TRANSLATIONS")
print("=" * 70)

predictions = []

# Check if line_end column exists, otherwise use a default
if 'line_end' in test_df.columns:
    total_lines = test_df['line_end'].max()
else:
    total_lines = len(test_df)  # fallback

for idx, row in test_df.iterrows():
    try:
        # Get line numbers if available, otherwise use index-based estimation
        line_start = row.get('line_start', idx + 1) if hasattr(row, 'get') else row['line_start'] if 'line_start' in test_df.columns else idx + 1
        line_end = row.get('line_end', idx + 1) if hasattr(row, 'get') else row['line_end'] if 'line_end' in test_df.columns else idx + 1
        
        # Extract segment based on line proportions
        segment = extract_translation_segment(
            translation=best_translation,
            line_start=line_start,
            line_end=line_end,
            total_lines=total_lines
        )
        
        predictions.append(segment)
        
        # Display results (only first 4 for brevity in submission)
        if len(predictions) <= 4:
            print(f"\n{'='*70}")
            print(f"Test ID: {row['id']} | Lines: {line_start}-{line_end}")
            print(f"{'='*70}")
            trans_text = row['transliteration'] if 'transliteration' in test_df.columns else 'N/A'
            print(f"\nInput ({len(str(trans_text))} chars):")
            print(str(trans_text)[:200] + "..." if len(str(trans_text)) > 200 else trans_text)
            print(f"\nOutput ({len(segment)} chars):")
            print(segment[:200] + "..." if len(segment) > 200 else segment)
    
    except Exception as e:
        print(f"Error processing row {idx}: {e}")
        # Use fallback translation
        fallback = best_translation[:200] if best_translation else "Translation not available."
        predictions.append(fallback)

print(f"\n{'='*70}")
print(f"Generated {len(predictions)} translations")
print(f"Best match similarity: {best_similarity:.1%}")

## 7. 📤 Create Submission

In [ ]:
# Create submission DataFrame
# IMPORTANT: Keep id as original type from test_df
submission = pd.DataFrame({
    'id': test_df['id'].values,  # Use .values to preserve original type
    'translation': predictions
})

# Validate and fix any issues
default_translation = best_translation[:300] if best_translation else "Ancient Akkadian commercial text."

for i in range(len(submission)):
    trans = submission.loc[i, 'translation']
    if trans is None or pd.isna(trans) or (isinstance(trans, str) and len(trans.strip()) == 0):
        print(f"Warning: Empty translation at index {i}, using fallback")
        submission.loc[i, 'translation'] = default_translation
    elif not isinstance(trans, str):
        submission.loc[i, 'translation'] = str(trans)

# Ensure no NaN
submission['translation'] = submission['translation'].fillna(default_translation)

# Display preview
print("Submission DataFrame:")
print(f"  Shape: {submission.shape}")
print(f"  Columns: {list(submission.columns)}")
print(f"  ID dtype: {submission['id'].dtype}")
print()
for idx, row in submission.iterrows():
    print(f"  id={row['id']}: {row['translation'][:80]}...")

In [ ]:
# Save submission - CRITICAL for competition scoring
output_path = '/kaggle/working/submission.csv' if os.path.exists('/kaggle/working') else 'submission.csv'

print(f"Saving to: {output_path}")

# Save with explicit settings matching competition format
submission.to_csv(output_path, index=False, encoding='utf-8')

# Verify the saved file
print("\n" + "="*60)
print("VERIFICATION")
print("="*60)

# Read back and verify
saved = pd.read_csv(output_path)
print(f"File saved successfully!")
print(f"  Rows: {len(saved)}")
print(f"  Columns: {list(saved.columns)}")

# Check format matches competition requirements
checks = []
checks.append(("Has 'id' column", 'id' in saved.columns))
checks.append(("Has 'translation' column", 'translation' in saved.columns))
checks.append(("Row count matches test", len(saved) == len(test_df)))
checks.append(("No NaN in translation", saved['translation'].notna().all()))
checks.append(("All translations non-empty", (saved['translation'].str.len() > 0).all()))

all_passed = True
for check_name, passed in checks:
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {check_name}")
    if not passed:
        all_passed = False

if not all_passed:
    raise ValueError("Submission validation failed!")

# Show actual CSV content (first few lines)
print("\nActual CSV content (first 500 chars):")
print("-"*60)
with open(output_path, 'r', encoding='utf-8') as f:
    content = f.read()
    print(content[:500])
    if len(content) > 500:
        print("...")
print("-"*60)
print(f"Total file size: {len(content)} bytes")

## 8. 📜 Full Translations Display

In [ ]:
print("=" * 70)
print("FINAL TRANSLATIONS")
print("=" * 70)

for _, row in submission.iterrows():
    print(f"\n[ID {row['id']}]")
    print("-" * 70)
    print(row['translation'])
    print()

---

## 📊 Method Summary

### Why Retrieval Works Better Than Neural MT

| Approach | Challenge | Result |
|----------|-----------|--------|
| 🤖 Neural MT (T5, mT5, NLLB) | 1,561 samples too small | BLEU < 8% |
| 🔍 TF-IDF Retrieval | Find similar training text | **87% match found!** |

> 💡 [Berkeley's CuneiTranslate project](https://www.ischool.berkeley.edu/projects/2024/cuneitranslate-unlocking-ancient-mesopotamian-knowledge) found that "no model achieved BLEU > 8%" due to dataset limitations. Our retrieval approach bypasses this!

### Why This Approach Works

| Factor | Explanation |
|--------|-------------|
| 📝 **Formulaic Language** | Ancient commercial documents follow standard patterns |
| 🎯 **High Similarity** | Test text has ~87% overlap with a training sample |
| ✂️ **Line-Based Segmentation** | Proportional extraction preserves meaning |
| 🔤 **Character N-grams** | Captures morphological patterns despite variations |

### 📚 Related Research

| Paper/Project | Method | Performance |
|---------------|--------|-------------|
| [PNAS Nexus 2023](https://academic.oup.com/pnasnexus/article/2/5/pgad096/7147349) | CNN + BPE | BLEU4: 37.47 |
| [Berkeley CuneiTranslate](https://www.ischool.berkeley.edu/projects/2024/cuneitranslate-unlocking-ancient-mesopotamian-knowledge) | T5, mT5, NLLB | BLEU < 8% |
| [AICC Project](https://praeclarum.org/2023/06/09/cuneiform.html) | T5 Fine-tuning | 130K translations |
| [Akkademia](https://github.com/gaigutherz/Akkademia) | BiLSTM | 97% accuracy |

### 🔮 Potential Improvements

1. **Hybrid Approach**: Combine retrieval with lexicon post-processing
2. **Data Augmentation**: Extract translations from `publications.csv` OCR
3. **Ensemble Methods**: TF-IDF + BM25 + Jaccard similarity
4. **Neural Reranking**: Use small LM to rerank retrieval candidates

---

### ⭐ If this notebook helped you, please upvote! ⭐

💬 Questions? Leave a comment!

---

## 9. 📊 Competition Data vs Extended Dataset

We created an **Extended Dataset** to help competitors. Here's how it compares to the competition data and how you can use it to improve your solution.

In [ ]:
# ============================================
# Load Extended Dataset (OPTIONAL - for analysis only)
# This section is NOT required for submission
# ============================================

EXTENDED_AVAILABLE = False

# Only try to load if we're not in submission mode (submission already saved above)
try:
    EXT_PATH = '/kaggle/input/old-assyrian-extended-corpus/'
    if os.path.exists(EXT_PATH):
        ext_corpus = pd.read_csv(f'{EXT_PATH}akkadian_corpus.csv')
        ext_dict = pd.read_csv(f'{EXT_PATH}akkadian_dictionary.csv')
        EXTENDED_AVAILABLE = True
        
        ext_parallel = ext_corpus[ext_corpus['has_translation'] == True]
        ext_mono = ext_corpus[ext_corpus['has_translation'] == False]
        
        print('Extended Dataset Loaded!')
        print(f'  Corpus: {len(ext_corpus):,} texts')
        print(f'  Dictionary: {len(ext_dict):,} entries')
    else:
        print('Extended dataset not added to notebook inputs.')
        print('To use: Click "Add Data" → Search "old-assyrian-extended-corpus"')
except Exception as e:
    print(f'Extended dataset not available: {e}')
    EXTENDED_AVAILABLE = False

In [ ]:
if EXTENDED_AVAILABLE:
    try:
        print('='*70)
        print('DATA COMPARISON: Competition vs Extended')
        print('='*70)
        
        print('\n┌────────────────────┬──────────────────┬──────────────────┐')
        print('│ Metric             │ Competition      │ Extended         │')
        print('├────────────────────┼──────────────────┼──────────────────┤')
        print(f'│ Parallel texts     │ {len(train_df):>16,} │ {len(ext_parallel):>16,} │')
        print(f'│ Monolingual texts  │ {0:>16,} │ {len(ext_mono):>16,} │')
        print(f'│ Total for TF-IDF   │ {len(train_df):>16,} │ {len(ext_corpus):>16,} │')
        print(f'│ Metadata columns   │ {len(train_df.columns):>16} │ {len(ext_corpus.columns):>16} │')
        print('├────────────────────┼──────────────────┼──────────────────┤')
        print(f'│ Data increase      │                  │ +{len(ext_mono)/len(train_df)*100:>13.0f}% │')
        print('└────────────────────┴──────────────────┴──────────────────┘')
    except Exception as e:
        print(f'Error in comparison: {e}')
else:
    print('Skipping extended dataset comparison (not loaded)')

### 9.1 Improved TF-IDF with Extended Corpus

Using the extended dataset (7,953 texts vs 1,561) can improve TF-IDF similarity matching!

In [ ]:
if EXTENDED_AVAILABLE:
    try:
        # Build TF-IDF on extended corpus
        print('Building TF-IDF on Extended Corpus...')
        print('='*70)
        
        # Use all texts for TF-IDF (parallel + monolingual)
        ext_vectorizer = TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(2, 6),
            max_features=25000,
            sublinear_tf=True
        )
        
        # Fit on ALL extended texts
        all_texts = ext_corpus['transliteration'].str.lower().tolist()
        ext_train_vectors = ext_vectorizer.fit_transform(all_texts)
        
        print(f'Original TF-IDF: trained on {len(train_df):,} texts')
        print(f'Extended TF-IDF: trained on {len(ext_corpus):,} texts')
        print(f'Vocabulary size: {len(ext_vectorizer.vocabulary_):,}')
    except Exception as e:
        print(f'Error building extended TF-IDF: {e}')
else:
    print('Skipping extended TF-IDF (dataset not loaded)')

In [ ]:
if EXTENDED_AVAILABLE:
    try:
        # Compare similarity scores
        print('Comparing Similarity Scores...')
        print('='*70)
        
        # Original: search in train_df only
        orig_test_vec = vectorizer.transform([full_test_text.lower()])
        orig_sims = cosine_similarity(orig_test_vec, train_vectors)[0]
        
        # Extended: search in parallel texts only (same translations available)
        ext_parallel_idx = ext_corpus[ext_corpus['has_translation'] == True].index
        ext_test_vec = ext_vectorizer.transform([full_test_text.lower()])
        ext_all_sims = cosine_similarity(ext_test_vec, ext_train_vectors)[0]
        ext_parallel_sims = ext_all_sims[ext_parallel_idx]
        
        print(f'\nBest match (Original TF-IDF):  {orig_sims.max():.4f} ({orig_sims.max()*100:.1f}%)')
        print(f'Best match (Extended TF-IDF):  {ext_parallel_sims.max():.4f} ({ext_parallel_sims.max()*100:.1f}%)')
        
        # Show improvement
        diff = ext_parallel_sims.max() - orig_sims.max()
        if diff > 0:
            print(f'\nImprovement: +{diff:.4f} ({diff*100:.2f}%)')
        else:
            print(f'\nDifference: {diff:.4f}')
    except Exception as e:
        print(f'Error comparing similarities: {e}')
else:
    print('Skipping similarity comparison (dataset not loaded)')

### 9.2 New Features Available in Extended Dataset

In [ ]:
if EXTENDED_AVAILABLE:
    try:
        print('NEW FEATURES in Extended Dataset')
        print('='*70)
        
        # Show the best match with extended metadata
        best_ext_idx = ext_parallel_sims.argmax()
        best_row = ext_parallel.iloc[best_ext_idx]
        
        print('\nBest matching text with EXTENDED metadata:')
        print('-'*70)
        print(f'Genre:           {best_row.get("genre_label", "N/A")}')
        print(f'CDLI ID:         {best_row.get("cdli_id", "N/A")}')
        print(f'Logogram count:  {best_row.get("logogram_count", "N/A")}')
        print(f'Gap count:       {best_row.get("gap_count", "N/A")}')
        print(f'Length ratio:    {best_row.get("length_ratio", "N/A")}')
        
        print('\nThese fields help you:')
        print('  - Filter by genre for domain-specific matching')
        print('  - Avoid texts with many gaps (damaged sections)')
        print('  - Estimate expected translation length')
    except Exception as e:
        print(f'Error showing features: {e}')
else:
    print('Skipping new features display (dataset not loaded)')

### 9.3 Logogram Dictionary: Understanding Sumerograms

In [ ]:
if EXTENDED_AVAILABLE:
    try:
        # Show logogram meanings
        logograms = ext_dict[ext_dict['entry_type'] == 'logogram']
        with_meaning = logograms[logograms['known_meaning'].notna()]
        
        print('LOGOGRAM DICTIONARY')
        print('='*70)
        print('\nSumerograms are Sumerian words written in ALL CAPS (e.g., KU.BABBAR).')
        print('Our dictionary provides meanings for the most common ones:')
        print('-'*70)
        
        for _, row in with_meaning.head(12).iterrows():
            freq = row.get('train_frequency', 0)
            meaning = row.get('known_meaning', 'N/A')
            form = row.get('form', 'N/A')
            print(f"  {form:15s} ({freq:>5,}x) = {meaning}")
        
        print('\nThis helps understand what the ancient texts are about!')
    except Exception as e:
        print(f'Error showing logograms: {e}')
else:
    print('Skipping logogram dictionary (dataset not loaded)')

### 9.4 How to Use Extended Dataset in Your Solution

In [ ]:
print('HOW TO USE EXTENDED DATASET')
print('='*70)
print('''
# 1. Add dataset to your notebook:
#    Click "Add Data" → Search "old-assyrian-extended-corpus"

# 2. Load:
corpus = pd.read_csv('/kaggle/input/old-assyrian-extended-corpus/akkadian_corpus.csv')
dictionary = pd.read_csv('/kaggle/input/old-assyrian-extended-corpus/akkadian_dictionary.csv')

# 3. Build better TF-IDF (use ALL 7,953 texts):
all_texts = corpus['transliteration'].str.lower()
vectorizer.fit(all_texts)  # Larger vocabulary!

# 4. Search only in parallel texts (with translations):
parallel = corpus[corpus['has_translation'] == True]
parallel_vectors = vectorizer.transform(parallel['transliteration'].str.lower())

# 5. Filter by quality:
clean_parallel = parallel[parallel['gap_count'] == 0]  # No damaged sections

# 6. Use logogram meanings for post-processing:
logograms = dictionary[dictionary['entry_type'] == 'logogram']
meanings = dict(zip(logograms['form'], logograms['known_meaning']))
''')

---

## Summary

| Approach | Data Used | Benefit |
|----------|-----------|--------|
| **Baseline** | Competition train.csv (1,561) | Quick start |
| **Extended** | akkadian_corpus.csv (7,953) | Better TF-IDF vocabulary |

---

### ⭐ If this notebook and extended dataset help you, please upvote! ⭐